# Profile one training step — Kaggle GPU

Answers what `docs/run_times.md` cannot: **where inside a step the time goes**,
what the real batch ceiling is, and what mixed precision buys. Trains nothing,
writes no checkpoint. See `decisions-pending.md` group E.

## Before you run

1. **Settings → Accelerator → GPU** (T4 or P100).
2. **Add data →** `tse-code` (from `kaggle_code.zip`). **The audio dataset is
   optional**: the batch sweep uses synthetic tensors of the right shape, because
   step time and peak memory depend on tensor SHAPES, not contents. Attach the
   ~2.9 GB audio dataset only if you want the data-loading probe (`RUN_LOADER`).
3. This is short — minutes, not hours — so an interactive session is fine. Use
   **Save & Run All (Commit)** anyway if you want the output kept.

## What each number decides

| number | what it settles |
|---|---|
| `fp32` vs `amp` s/step and GB | whether to adopt mixed precision (E4 lever 1) |
| the largest batch that does not OOM | whether checkpointing is needed (E4 lever 3) |
| `band_rnn` % of forward | confirms E2 on real hardware |
| `subband_norm` + `estimator` % | whether the 32-band loops are worth touching |
| loader s/batch vs s/step | whether audio compression buys anything (E5) |

CPU profiling measured band loops at 4.0 % and loading at 6.5 % — **both
non-bottlenecks**. The T4 has kernel-launch overhead that CPU does not, so the
band-loop share is the one figure that could differ here.


In [ ]:
# ============================== KNOBS ==============================
BATCHES     = [3, 6, 12]   # trials/step. Each runs in its OWN subprocess, so an
                           # OOM kills the child and the sweep continues.
STEPS       = 8            # timed steps per candidate, after 3 warmup
AMP_ONLY    = False       # True = skip fp32 and find the AMP batch ceiling.
                          # Needed as a SECOND run: fp32 OOMs first and kills
                          # the process before AMP is reached.
CHUNK_S     = None        # None = use the config's 4.0 (T=503, only batch 4 aligns).
                          # 4.008 gives T=504, a multiple of 8, so EVERY batch hits
                          # the fp16 tensor-core kernel. See E3f.
RUN_LOADER  = False        # True needs the audio dataset attached below
SPLIT       = "sir0"

# HINTS ONLY -- the next cell finds these by content if they are wrong,
# so a different account username or mount depth does not matter.
CODE_DIR = "/kaggle/input/datasets/grantbooysen/tse-code"
DATA_DIR = "/kaggle/input/datasets/grantbooysen/tse-sir0-audio"   # only if RUN_LOADER
# ===================================================================

WORK = "/kaggle/working"
REPO = f"{WORK}/repo"      # writable copy; /kaggle/input is read-only
RES  = f"{WORK}/profile"


In [ ]:
# --- environment + GPU ----------------------------------------------------
import os, sys, subprocess, shutil, re, json
from pathlib import Path

# Set BEFORE torch initialises CUDA or it has no effect. Same setting the
# training notebook uses: the OOM traceback reports GiB "reserved but
# unallocated", which is fragmentation rather than real demand.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"  device: {p.name}  {p.total_memory/2**30:.1f} GiB")
else:
    raise SystemExit("NO GPU -- Settings -> Accelerator -> GPU. "
                     "The CPU numbers are already in decisions-pending.md E3b.")

try:
    import soundfile
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile"], check=True)

# --- locate the inputs, by CONTENT not by name ---------------------------
# The hardcoded knobs carry an account username, so they break on any account
# but the one they were written on. Kaggle also mounts datasets at more than one
# depth depending on how they were attached. Searching for a file we know must
# be there is stable against both.
def find_input(marker, configured):
    """Directory under /kaggle/input containing `marker`. Tries the configured
    path first so an explicit override always wins."""
    if configured and (Path(configured) / marker).exists():
        return Path(configured)
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    seen = []
    for depth in ("*", "*/*", "*/*/*"):
        for cand in sorted(root.glob(depth)):
            if cand.is_dir():
                seen.append(cand)
                if (cand / marker).exists():
                    return cand
    find_input.seen = seen
    return None

# profile_step.py is the file this notebook exists to run -- if it is missing the
# bundle is stale, which is a different error than a wrong path.
CODE = find_input("scripts/profile_step.py", CODE_DIR)
if CODE is None:
    listing = "\n".join(f"    {d}" for d in getattr(find_input, "seen", [])) or "    (nothing mounted)"
    raise SystemExit(
        "Could not find the code dataset (no scripts/profile_step.py under "
        "/kaggle/input).\nMounted directories:\n" + listing +
        "\n\nEither: attach the tse-code dataset (Add Input ->), or if it IS "
        "attached above,\nre-run make_kaggle_bundle.py and re-upload -- an older "
        "kaggle_code.zip has no profile_step.py.")
print(f"code: {CODE}")

DATA = None
if RUN_LOADER:
    DATA = find_input("data/manifests", DATA_DIR)
    if DATA is None:
        raise SystemExit("RUN_LOADER=True but no dataset with data/manifests found. "
                         "Attach the audio dataset, or set RUN_LOADER=False -- the "
                         "batch sweep uses synthetic tensors and needs no audio.")
    print(f"data: {DATA}")
else:
    print("data: (not needed -- the batch sweep uses synthetic tensors)")


In [ ]:
# --- stage the code somewhere writable ------------------------------------
# /kaggle/input is read-only and src.run_log writes repo_root/docs/run_times.md,
# so the code cannot run in place.
if Path(REPO).exists():
    shutil.rmtree(REPO)
# NO ignore= here: ignore_patterns("data") matches ANY directory named `data` at
# any depth and silently drops src/data/. Same trap as the training notebook.
shutil.copytree(CODE, REPO)
Path(RES).mkdir(parents=True, exist_ok=True)

chk = subprocess.run(
    [sys.executable, "-c", "import sys; sys.path.insert(0, '.'); "
     "import scripts.profile_step, src.models.bsrnn, src.models.losses"],
    cwd=REPO, capture_output=True, text=True)
if chk.returncode:
    raise SystemExit(f"staged code at {REPO} does not import:\n{chk.stderr}")
print("staged code imports OK")

if not (Path(REPO) / "scripts/profile_step.py").exists():
    raise SystemExit("scripts/profile_step.py missing from the code bundle -- "
                     "re-run make_kaggle_bundle.py and re-upload tse-code.")


In [ ]:
# --- batch sweep ----------------------------------------------------------
# One SUBPROCESS per batch size. A caught OutOfMemoryError leaves the allocator
# in a poor state and the failed graph can stay reachable, so retrying
# in-process measures the wrong thing; process exit is the only reliable free.
RE_F32  = re.compile(r"fp32\s+([\d.]+) s/step\s+peak\s+([\d.]+) GB")
RE_AMP  = re.compile(r"amp\s+([\d.]+) s/step\s+peak\s+([\d.]+) GB")
RE_GAIN = re.compile(r"AMP\s+([\d.]+)x faster,\s+([\d.]+)x less memory")
RE_MOD  = re.compile(r"^\s+(stft|tfmap|subband_norm|separator|estimator)\s+([\d.]+)s\s+([\d.]+)%")
RE_RNN  = re.compile(r"(time_rnn|band_rnn)\s+seq=\S+\s+batch=\S+\s+([\d.]+)s\s+([\d.]+)%")

rows, raw = [], {}
for run_i, b in enumerate(BATCHES):
    print(f"\n{'='*66}\nbatch {b}\n{'='*66}", flush=True)
    r = subprocess.run(
        [sys.executable, "-u", "scripts/profile_step.py", "--batch", str(b),
         "--steps", str(STEPS)] + (["--amp-only"] if AMP_ONLY else [])
          + (["--chunk-s", str(CHUNK_S)] if CHUNK_S else []),
        cwd=REPO, capture_output=True, text=True)
    out = r.stdout + r.stderr
    print(out, flush=True)
    raw[(run_i, b)] = out
    if "OutOfMemoryError" in out or "CUDA out of memory" in out:
        rows.append(dict(run=run_i, batch=b, status="OOM"))
        mode = "fp16/AMP" if AMP_ONLY else "fp32"
        print(f"--> batch {b} OOM in {mode}: that is the ceiling for this precision.")
        continue
    if r.returncode != 0:
        rows.append(dict(run=run_i, batch=b, status=f"FAILED rc={r.returncode}"))
        continue
    row = dict(run=run_i, batch=b, status="ok")
    if (m := RE_F32.search(out)):  row.update(fp32_s=float(m[1]), fp32_gb=float(m[2]))
    if (m := RE_AMP.search(out)):  row.update(amp_s=float(m[1]),  amp_gb=float(m[2]))
    if (m := RE_GAIN.search(out)): row.update(speedup=float(m[1]), mem_x=float(m[2]))
    for m in RE_MOD.finditer(out):  row[f"{m[1]}_pct"] = float(m[3])
    for m in RE_RNN.finditer(out):  row[f"{m[1]}_pct"] = float(m[3])
    rows.append(row)

Path(RES, "profile_raw.txt").write_text("\n\n".join(f"### run {i} batch {b}\n{o}" for (i, b), o in raw.items()))


In [ ]:
# --- data loading alone (optional) ---------------------------------------
# Its own invocation: DataLoader workers fork the parent, so building the model
# in the same process duplicates its memory into every worker.
loader_s = None
if RUN_LOADER:
    r = subprocess.run(
        [sys.executable, "-u", "scripts/profile_step.py", "--loader-only",
         "--split", SPLIT, "--steps", "8",
         "--data-root", str(DATA / "data"),
         "--manifest-dir", str(DATA / "data/manifests")],
        cwd=REPO, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if (m := re.search(r"([\d.]+) s/batch", r.stdout)):
        loader_s = float(m[1])
else:
    print("RUN_LOADER=False -- skipped. CPU measured 0.382 s/batch single-threaded")
    print("(worst case, 6.5% of a 5.84 s step), so loading was not the bottleneck.")


In [ ]:
# --- summary --------------------------------------------------------------
import pandas as pd
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(Path(RES, "profile_summary.csv"), index=False)

ok = df[df.status == "ok"] if "status" in df else df
if len(ok):
    top = ok.iloc[0]
    print("\n--- what this says ---")
    if "speedup" in ok and pd.notna(top.get("speedup")):
        print(f"AMP at batch {int(top.batch)}: {top.speedup:.2f}x faster, "
              f"{top.mem_x:.2f}x less memory.")
        print("  -> adopt it if >1.3x on either axis (E4 lever 1). Remember the")
        print("     loss must stay fp32: 1e-12 epsilons underflow in fp16.")
    if "band_rnn_pct" in ok and pd.notna(top.get("band_rnn_pct")):
        loops = (top.get("subband_norm_pct", 0) or 0) + (top.get("estimator_pct", 0) or 0)
        print(f"band_rnn {top.band_rnn_pct:.1f}% of forward, time_rnn "
              f"{top.get('time_rnn_pct', float('nan')):.1f}%; 32-band loops {loops:.1f}%.")
        print("  -> CPU measured loops at 4.0%. If they are still <10% here, E4")
        print("     lever 5 (grouped convs) stays off the list.")
    oom = df[df.status == "OOM"] if "status" in df else df.iloc[0:0]
    if len(oom):
        print(f"fp32 ceiling: batch {int(oom.batch.min())} OOMs -> checkpointing "
              f"(E4 lever 3) is what unlocks a bigger batch.")
    else:
        print(f"No OOM up to batch {int(ok.batch.max())} -- raise BATCHES and rerun.")
if loader_s:
    ref = ok.fp32_s.iloc[0] if len(ok) and "fp32_s" in ok else float("nan")
    print(f"loader {loader_s:.3f} s/batch vs {ref:.3f} s/step "
          f"-> {'STARVING the GPU' if loader_s > ref else 'not the bottleneck'}")

print(f"\nsaved: {RES}/profile_summary.csv and profile_raw.txt")
print("Paste the summary into decisions-pending.md E3b as the GPU column.")
